<a href="https://colab.research.google.com/github/printf-sourav/MediSync/blob/Ml/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

--- LOADING DATASET ---
Columns: Index(['id', 'facility_level', 'facility_id', 'region_type', 'has_pharmacist',
       'pharmacy_staff_count', 'lmis_type', 'lmis_functional',
       'cold_chain_functional', 'storage_adequate', 'supervised_last_quarter',
       'drug_name', 'drug_category', 'formulation', 'ven_classification',
       'unit_cost_usd', 'year', 'quarter', 'available_on_survey_day',
       'stocked_out_in_last_6m', 'stockout_days_last_6m',
       'stockout_episodes_last_6m', 'longest_stockout_days',
       'stockout_cause_primary', 'stockout_cause_secondary',
       'order_placed_on_time', 'last_delivery_days_ago', 'lead_time_days',
       'delivery_complete', 'order_fill_rate_pct', 'expired_stock_found',
       'temperature_excursion', 'damaged_stock_found', 'stock_card_up_to_date',
       'physical_count_matches_records', 'FEFO_compliance',
       'patients_turned_away', 'referred_to_private_pharmacy',
       'treatment_substituted', 'treatment_delayed', 'LMIS_report_subm

KeyError: 'stockout_days'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from datasets import load_dataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset, random_split

# =========================================================
# 1. LOAD DATA
# =========================================================
dataset = load_dataset(
    "electricsheepafrica/essential-medicines-stockouts",
    split="train[:5000]"
)

df = pd.DataFrame(dataset)

print("Columns:", df.columns)

# =========================================================
# 2. AUTO SELECT NUMERIC COLUMNS
# =========================================================
numeric_df = df.select_dtypes(include=['number'])

if numeric_df.shape[1] < 1:
    raise ValueError("No numeric columns found in dataset!")

print("Numeric Columns:", numeric_df.columns)

# Use first column as feature
X = numeric_df.iloc[:, :-1].values if numeric_df.shape[1] > 1 else numeric_df.values

# Create artificial target (for demo)
y = (numeric_df.iloc[:, 0] > numeric_df.iloc[:, 0].median()).astype(float).values.reshape(-1, 1)

# =========================================================
# 3. PREPROCESS
# =========================================================
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

# =========================================================
# 4. MODEL
# =========================================================
class Model(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.out = nn.Linear(16, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

model = Model(input_dim=X.shape[1])

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

# =========================================================
# 5. TRAIN
# =========================================================
for epoch in range(30):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for vx, vy in val_loader:
            probs = torch.sigmoid(model(vx))
            pred = (probs > 0.5).int()

            preds.extend(pred.numpy().flatten())
            targets.extend(vy.numpy().flatten())

    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds)

    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f} | Acc: {acc:.2f} | F1: {f1:.2f}")

Columns: Index(['id', 'facility_level', 'facility_id', 'region_type', 'has_pharmacist',
       'pharmacy_staff_count', 'lmis_type', 'lmis_functional',
       'cold_chain_functional', 'storage_adequate', 'supervised_last_quarter',
       'drug_name', 'drug_category', 'formulation', 'ven_classification',
       'unit_cost_usd', 'year', 'quarter', 'available_on_survey_day',
       'stocked_out_in_last_6m', 'stockout_days_last_6m',
       'stockout_episodes_last_6m', 'longest_stockout_days',
       'stockout_cause_primary', 'stockout_cause_secondary',
       'order_placed_on_time', 'last_delivery_days_ago', 'lead_time_days',
       'delivery_complete', 'order_fill_rate_pct', 'expired_stock_found',
       'temperature_excursion', 'damaged_stock_found', 'stock_card_up_to_date',
       'physical_count_matches_records', 'FEFO_compliance',
       'patients_turned_away', 'referred_to_private_pharmacy',
       'treatment_substituted', 'treatment_delayed', 'LMIS_report_submitted',
       'LMIS_rep

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

model.eval()
preds, targets = [], []

with torch.no_grad():
    for vx, vy in val_loader:   # or test_loader
        logits = model(vx)
        probs = torch.sigmoid(logits)   # convert to probability

        pred = (probs > 0.5).int()      # threshold

        preds.extend(pred.cpu().numpy().flatten())
        targets.extend(vy.cpu().numpy().flatten())

# Metrics
acc = accuracy_score(targets, preds)
f1 = f1_score(targets, preds, zero_division=0)
precision = precision_score(targets, preds, zero_division=0)
recall = recall_score(targets, preds, zero_division=0)

print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

Accuracy : 0.9910
F1 Score : 0.9914
Precision: 0.9923
Recall   : 0.9904
